In [86]:
from sagemaker.workflow.function_step import step
from sagemaker.workflow.pipeline import Pipeline
import sagemaker
from sagemaker.workflow.parameters import ParameterInteger
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep
import boto3

In [87]:
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

region = boto3.Session().region_name
print(f"Region: {region}")
user = "grupo5"

TRACKING_SERVER = "mlflow-tracking-server-grupo5"
default_bucket = "s3-mlflow-artifacts-mlflow-tracking-server-grupo5-01"
default_prefix = f"sagemaker/clients-attrition/{user}"
default_path = default_bucket + "/" + default_prefix
athena_database = "glue-catalog-database-utec-bank-01"

role = sagemaker.get_execution_role()

Region: us-east-1


In [88]:
sagemaker_session = sagemaker.Session(default_bucket=default_bucket,
                                      default_bucket_prefix=default_prefix)
# #Pipeline configuration
instance_type = "ml.m5.large"
pipeline_name = f"pipeline-train-{user}"
model_name = f"model-attrition-{user}"

# MLFlow configuration
tracking_server_arn = f"arn:aws:sagemaker:{region}:{account_id}:mlflow-tracking-server/{TRACKING_SERVER}"
experiment_name = f"pipeline-train-exp-{user}"

In [89]:
edad_init = ParameterInteger(name="EdadInit")
edad_end = ParameterInteger(name="EdadEnd")

In [90]:
# jalar datos para el entrenamiento
@step(
    name="DataPull",
    instance_type=instance_type,
    dependencies="./data_pull_requirements.txt"
)
def data_pull(experiment_name: str, run_name: str) -> tuple[str, str]:
  import awswrangler as wr
  import mlflow

  mlflow.set_tracking_uri(tracking_server_arn)
  mlflow.set_experiment(experiment_name)
  TARGET_COL = "attrition"
  query = """
SELECT
  -- columnas de clientes
  c.id_correlativo AS id_cliente,
  c.codmes AS codmes_cliente,
  c.flg_bancarizado,
  c.rang_ingreso,
  c.flag_lima_provincia,
  c.edad,
  c.antiguedad,
  c.attrition,
  c.rang_sdo_pasivo_menos0,
  c.sdo_activo_menos0,
  c.sdo_activo_menos1,
  c.sdo_activo_menos2,
  c.sdo_activo_menos3,
  c.sdo_activo_menos4,
  c.sdo_activo_menos5,
  c.flg_seguro_menos0,
  c.flg_seguro_menos1,
  c.flg_seguro_menos2,
  c.flg_seguro_menos3,
  c.flg_seguro_menos4,
  c.flg_seguro_menos5,
  c.rang_nro_productos_menos0,
  c.flg_nomina,
  c.nro_acces_canal1_menos0,
  c.nro_acces_canal1_menos1,
  c.nro_acces_canal1_menos2,
  c.nro_acces_canal1_menos3,
  c.nro_acces_canal1_menos4,
  c.nro_acces_canal1_menos5,
  c.nro_acces_canal2_menos0,
  c.nro_acces_canal2_menos1,
  c.nro_acces_canal2_menos2,
  c.nro_acces_canal2_menos3,
  c.nro_acces_canal2_menos4,
  c.nro_acces_canal2_menos5,
  c.nro_acces_canal3_menos0,
  c.nro_acces_canal3_menos1,
  c.nro_acces_canal3_menos2,
  c.nro_acces_canal3_menos3,
  c.nro_acces_canal3_menos4,
  c.nro_acces_canal3_menos5,
  c.nro_entid_ssff_menos0,
  c.nro_entid_ssff_menos1,
  c.nro_entid_ssff_menos2,
  c.nro_entid_ssff_menos3,
  c.nro_entid_ssff_menos4,
  c.nro_entid_ssff_menos5,
  c.flg_sdo_otssff_menos0,
  c.flg_sdo_otssff_menos1,
  c.flg_sdo_otssff_menos2,
  c.flg_sdo_otssff_menos3,
  c.flg_sdo_otssff_menos4,
  c.flg_sdo_otssff_menos5,

  -- columnas de requerimientos (con alias si es necesario)
  r.codmes AS codmes_requerimiento,
  r.tipo_requerimiento2,
  r.dictamen,
  r.producto_servicio_2,
  r.submotivo_2

FROM "glue-catalog-database-utec-bank-01"."clientes" AS c
LEFT JOIN "glue-catalog-database-utec-bank-01"."requerimientos" AS r
  ON c.id_correlativo = r.id_correlativo
    """
  train_s3_path = f"s3://{default_path}/data_train/trained_clientes_pipeline.csv"
  with mlflow.start_run(run_name=run_name) as run:
    run_id = run.info.run_id
    with mlflow.start_run(run_name="DataPull", nested=True):
      df = wr.athena.read_sql_query(sql=query, database=athena_database)
      df.to_csv(train_s3_path, index=False)
      mlflow.log_input(
          mlflow.data.from_pandas(df, train_s3_path,
                                  targets=TARGET_COL),
          context="DataPull"
      )
  return train_s3_path, experiment_name, run_id

In [91]:
@step(
    name="DataPreprocessing",
    instance_type="ml.m5.large",
    dependencies="./Clean_data_requirements.txt"
)
def preprocess_model(train_s3_path: str, experiment_name: str, run_id) -> tuple[str, str, str]:
    import pandas as pd
    import mlflow
    import awswrangler as wr
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import LabelEncoder

    target_col = "attrition"
    train_s3_path2 = f"s3://{default_path}/data_train/trained_clientes_clean.csv"

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name="data_clean") as run:
        run_id = run.info.run_id

        with mlflow.start_run(run_name="DataPull", nested=True):
           #df = wr.athena.read_sql_query(sql=query, database=athena_database)
            df = pd.read_csv(train_s3_path)

            # Limpieza de columnas con muchos nulos
            df.drop(columns=[
                'tipo_requerimiento2', 'dictamen',
                'producto_servicio_2', 'submotivo_2'
            ], inplace=True, errors='ignore')

            # Imputación
            cat_imputer = SimpleImputer(strategy='most_frequent')
            df[['rang_ingreso', 'flag_lima_provincia']] = cat_imputer.fit_transform(
                df[['rang_ingreso', 'flag_lima_provincia']]
            )

            num_imputer = SimpleImputer(strategy='median')
            df[['edad', 'antiguedad']] = num_imputer.fit_transform(
                df[['edad', 'antiguedad']]
            )

            # Codificación
            cat_cols = df.select_dtypes(include='object').columns.tolist()
            for col in cat_cols:
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col])

            # Selección de features
            selected_features = [
                'rang_sdo_pasivo_menos0', 'edad', 'antiguedad', 'rang_ingreso',
                'nro_acces_canal3_menos0', 'nro_acces_canal3_menos1', 'nro_acces_canal3_menos2',
                'nro_acces_canal3_menos3', 'rang_nro_productos_menos0', 'nro_entid_ssff_menos0',
                'nro_acces_canal3_menos4', 'nro_entid_ssff_menos5', 'sdo_activo_menos0',
                'nro_acces_canal3_menos5', 'nro_entid_ssff_menos4'
            ]

            df_final = df[selected_features + [target_col]]

            # Guardar en S3
            df_final.to_csv(train_s3_path2, index=False)

            # Loguear en MLflow
            mlflow.log_input(
                mlflow.data.from_pandas(df_final, train_s3_path, targets=target_col),
                context="DataPull"
            )

    return train_s3_path2, experiment_name, run_id


In [92]:
@step(
    name="AttritionModelTraining",
    instance_type="ml.m5.xlarge",
    dependencies="./model_training_requirements.txt"
)
def model_training(train_s3_path: str, experiment_name: str,
                   run_id: str) -> tuple[str, str, str, str]:
  import pandas as pd
  import mlflow
  from sklearn.model_selection import train_test_split, RandomizedSearchCV
  from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
  from lightgbm import LGBMClassifier
  from xgboost import XGBClassifier
  from sklearn.ensemble import RandomForestClassifier
  import numpy as np
  import joblib

  TARGET_COL = "attrition"
  SEED = 42
  TRAIN_SPLIT = 0.7

  mlflow.set_tracking_uri(tracking_server_arn)
  mlflow.set_experiment(experiment_name)

  df = pd.read_csv(train_s3_path)

  X = df.drop(columns=[TARGET_COL])
  y = df[TARGET_COL]

  X_train, X_test, y_train, y_test = train_test_split(
      X, y,
      train_size=TRAIN_SPLIT,
      random_state=SEED,
      stratify=y
  )

  scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

  # --- LightGBM Training ---
  mlflow.set_experiment("LightGBM_Attrition_Model")
  with mlflow.start_run(run_name="lightgbm_attrition") as run:
      lgbm_run_id = run.info.run_id
      search = RandomizedSearchCV(
          LGBMClassifier(random_state=SEED),
          param_distributions={
              "n_estimators": [100, 200],
              "max_depth": [3, 4, 5],
              "learning_rate": [0.01, 0.05, 0.1],
              "subsample": [0.6, 0.8, 1.0],
              "colsample_bytree": [0.6, 0.8, 1.0]
          },
          n_iter=10, scoring='f1', cv=3, verbose=1, n_jobs=-1, random_state=SEED
      )

      search.fit(X_train, y_train)
      best_model = search.best_estimator_

      y_pred = best_model.predict(X_test)
      y_proba = best_model.predict_proba(X_test)[:, 1]

      report = classification_report(y_test, y_pred, output_dict=True)
      roc_auc = roc_auc_score(y_test, y_proba)

      mlflow.log_params(search.best_params_)
      mlflow.log_metric("roc_auc", roc_auc)
      mlflow.log_metric("f1_score", report['1']['f1-score'])
      mlflow.log_metric("recall", report['1']['recall'])
      mlflow.log_metric("precision", report['1']['precision'])
      mlflow.log_metric("train_size", len(X_train))
      mlflow.log_metric("test_size", len(X_test))
      mlflow.log_metric("attrition_rate", y.mean())
      mlflow.log_metric("class_imbalance_ratio", len(y[y == 0]) / len(y[y == 1]))
      mlflow.sklearn.log_model(best_model, "model")

      FEATURES = X_train.columns.tolist()
      feature_importance = pd.DataFrame({
          'feature': FEATURES,
          'importance': best_model.feature_importances_
      }).sort_values('importance', ascending=False)
      mlflow.log_text(feature_importance.to_string(), "lightgbm_feature_importance.txt")

  # --- XGBoost Training with autolog ---
  mlflow.set_experiment("experiment_xgboost_autolog")
  with mlflow.start_run() as run:
      xgb_run_id = run.info.run_id
      mlflow.xgboost.autolog(
          log_input_examples=True,
          log_model_signatures=True,
          log_models=True,
          log_datasets=True,
          model_format="pyfunc"
      )

      model = XGBClassifier(
          objective='binary:logistic',
          scale_pos_weight=scale_pos_weight,
          random_state=SEED,
          use_label_encoder=False,
          eval_metric='logloss'
      )

      param_dist = {
          "n_estimators": [100, 200],
          "max_depth": [3, 4, 5],
          "learning_rate": [0.01, 0.05, 0.1],
          "subsample": [0.6, 0.8, 1.0],
          "colsample_bytree": [0.6, 0.8, 1.0],
          "gamma": [0, 0.1, 0.2]
      }

      search = RandomizedSearchCV(
          model, param_distributions=param_dist,
          n_iter=10, scoring='f1', cv=3, verbose=1, n_jobs=-1, random_state=SEED
      )

      search.fit(X_train, y_train)
      best_model = search.best_estimator_

      y_pred = best_model.predict(X_test)
      y_proba = best_model.predict_proba(X_test)[:, 1]

      f1 = f1_score(y_test, y_pred)
      auc = roc_auc_score(y_test, y_proba)
      report = classification_report(y_test, y_pred, output_dict=True)
      mlflow.log_metric("f1_score", f1)
      mlflow.log_metric("roc_auc", auc)
      mlflow.log_param("scale_pos_weight", scale_pos_weight)

      FEATURES = X_train.columns.tolist()
      feature_importance = pd.DataFrame({
          'feature': FEATURES,
          'importance': best_model.feature_importances_
      }).sort_values('importance', ascending=False)

      mlflow.log_text(feature_importance.to_string(), "xgboost_feature_importance.txt")

      joblib.dump(best_model, "xgb_model.joblib")
      mlflow.log_artifact("xgb_model.joblib")

  # --- Random Forest Training ---
  mlflow.set_experiment("RandomForest_Attrition_Model")
  with mlflow.start_run(run_name="randomforest_attrition") as run:
      rf_run_id = run.info.run_id
      rf_model = RandomForestClassifier(
          n_estimators=100,
          max_depth=5,
          class_weight='balanced',
          random_state=SEED
      )
      rf_model.fit(X_train, y_train)

      y_pred = rf_model.predict(X_test)
      y_proba = rf_model.predict_proba(X_test)[:, 1]

      f1 = f1_score(y_test, y_pred)
      auc = roc_auc_score(y_test, y_proba)
      report = classification_report(y_test, y_pred, output_dict=True)

      mlflow.log_metric("f1_score", f1)
      mlflow.log_metric("roc_auc", auc)
      mlflow.log_metric("precision", report['1']['precision'])
      mlflow.log_metric("recall", report['1']['recall'])
      mlflow.log_metric("train_size", len(X_train))
      mlflow.log_metric("test_size", len(X_test))

      mlflow.sklearn.log_model(rf_model, "model")

      FEATURES = X_train.columns.tolist()
      feature_importance = pd.DataFrame({
          'feature': FEATURES,
          'importance': rf_model.feature_importances_
      }).sort_values('importance', ascending=False)

      mlflow.log_text(feature_importance.to_string(), "randomforest_feature_importance.txt")

  test_s3_path = f"s3://{default_path}/test_data/attrition_test.csv"
  df_test = pd.concat([X_test, y_test], axis=1)
  df_test.to_csv(test_s3_path, index=False)

  return test_s3_path, experiment_name, lgbm_run_id, xgb_run_id, rf_run_id


In [ ]:
@step(
    name="ModelEvaluation",
    instance_type=instance_type,
    dependencies="./model_evaluation_requirements.txt"
)
def evaluate(
    test_s3_path: str,
    experiment_name: str,
    lgbm_run_id: str,
    xgb_run_id: str,
    rf_run_id: str,
) -> dict:
    import mlflow
    import pandas as pd

    TARGET_COL = "attrition"
    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)
    test_df = pd.read_csv(test_s3_path)

    model_runs = {
        "lightgbm": lgbm_run_id,
        "xgboost": xgb_run_id,
        "randomforest": rf_run_id
    }

    scores = {}

    with mlflow.start_run(run_name="ModelEvaluation", nested=True):
        for name, run_id in model_runs.items():
            model = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")
            results = mlflow.evaluate(
                model=model,
                data=test_df,
                targets=TARGET_COL,
                model_type="classifier",
                evaluators=["default"],
            )
            f1 = results.metrics["f1_score"]
            scores[name] = {"f1_score": f1, "run_id": run_id}

        # Elegir el mejor modelo por f1_score
        best_model_name = max(scores, key=lambda m: scores[m]["f1_score"])
        best_model_run_id = scores[best_model_name]["run_id"]
        best_f1 = scores[best_model_name]["f1_score"]

        mlflow.log_param("best_model", best_model_name)
        mlflow.log_metric("best_f1_score", best_f1)

        # Realizar predicción con el mejor modelo
        best_model = mlflow.pyfunc.load_model(f"runs:/{best_model_run_id}/model")
        y_pred = best_model.predict(test_df.drop(columns=[TARGET_COL]))
        test_df["prediction"] = y_pred

        # (Opcional) guardar predicciones en CSV
        predictions_path = f"s3://{default_path}/predictions/{best_model_name}_predictions.csv"
        test_df.to_csv(predictions_path, index=False)

    return {
        "best_model": best_model_name,
        "best_model_run_id": best_model_run_id,       
        "best_f1_score": best_f1,
        "predictions_path": predictions_path
    }


In [ ]:
@step(
    name="ModelRegistration",
    instance_type=instance_type,
    dependencies="./model_registration_requirements.txt"
)
def register(
    model_name: str,
    experiment_name: str,
    run_id: str,
    training_run_id: str,
) -> str:
    import mlflow

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelRegistration", nested=True) as nested_run:
            result = mlflow.register_model(
                model_uri=f"runs:/{training_run_id}/model",
                name=model_name
            )
            mlflow.log_param("registered_model_name", model_name)
            mlflow.log_param("registered_model_version", result.version)
            print(f"✅ Modelo registrado: {result.name} (versión {result.version})")
            return result.version 

In [95]:
data_pull_step = data_pull(experiment_name=experiment_name,
                           run_name=ExecutionVariables.PIPELINE_EXECUTION_ID)

data_preprocessing_step = preprocess_model(
    train_s3_path=data_pull_step[0],
    experiment_name=data_pull_step[1],
    run_id=data_pull_step[2] 
)

model_training_step = model_training(train_s3_path=data_preprocessing_step[0],
                                     experiment_name=data_preprocessing_step[1],
                                     run_id=data_preprocessing_step[2])

evaluate_step = evaluate(
    test_s3_path=model_training_step[0],
    experiment_name=model_training_step[1],
    lgbm_run_id=model_training_step[2],
    xgb_run_id=model_training_step[3],
    rf_run_id=model_training_step[4]
)

conditional_register_step = ConditionStep(
    name="ConditionalRegister",
    conditions=[
        ConditionGreaterThanOrEqualTo(
            left=evaluate_step["best_f1_score"],
            right=0.4,
        )
    ],
    if_steps=[
        register(
            model_name=evaluate_step["best_model"],
            experiment_name=model_training_step[1],
            run_id=data_pull_step[2] ,
            training_run_id=evaluate_step["best_model_run_id"],
        )
    ],
    else_steps=[FailStep(name="Fail",
                         error_message="Model performance is not good enough")]
)

In [96]:
steps = [
    data_pull_step,
    data_preprocessing_step,
    model_training_step,
    conditional_register_step
]
pipeline = Pipeline(name=pipeline_name,
                    steps=steps)
pipeline.upsert(role_arn=role)

2025-06-27 00:52:05,346 sagemaker.remote_function INFO     Uploading serialized function code to s3://sagemaker-us-east-1-189651425406/pipeline-train-grupo5/DataPull/2025-06-27-00-52-05-072/function
2025-06-27 00:52:05,413 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://sagemaker-us-east-1-189651425406/pipeline-train-grupo5/DataPull/2025-06-27-00-52-05-072/arguments
2025-06-27 00:52:05,719 sagemaker.remote_function INFO     Copied dependencies file at './data_pull_requirements.txt' to '/tmp/tmp_w9ypblm/data_pull_requirements.txt'
2025-06-27 00:52:05,755 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://sagemaker-us-east-1-189651425406/pipeline-train-grupo5/DataPull/2025-06-27-00-52-05-072/pre_exec_script_and_dependencies'
2025-06-27 00:52:05,758 sagemaker.remote_function INFO     Uploading serialized function code to s3://sagemaker-us-east-1-189651425406/pipeline-train-grupo5/DataPreprocessing/202

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:189651425406:pipeline/pipeline-train-grupo5',
 'ResponseMetadata': {'RequestId': '443f51c1-08dd-4f8f-ab02-e1f3b9bce76a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '443f51c1-08dd-4f8f-ab02-e1f3b9bce76a',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '89',
   'date': 'Fri, 27 Jun 2025 00:52:08 GMT'},
  'RetryAttempts': 0}}

In [97]:
import random
import string
from datetime import datetime

def generate_random_string(length=8):
    return ''.join(random.choices(string.ascii_lowercase + string.digits, k=length))

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
random_suffix = generate_random_string(4)

pipeline.start(
    execution_display_name=f"exec-{timestamp}-{random_suffix}",
    execution_description=f"Pipeline run at {timestamp}"
)

_PipelineExecution(arn='arn:aws:sagemaker:us-east-1:189651425406:pipeline/pipeline-train-grupo5/execution/kl1okeynd1tu', sagemaker_session=<sagemaker.session.Session object at 0x7f5e48b348c0>)

In [ ]:
scheduler_client = boto3.client('scheduler', region_name=region)

def create_eventbridge_schedule():
    """Crear un Schedule en EventBridge Scheduler"""
    
    schedule_name = f"pipeline-schedule-{user}"
    
    try:
        response = scheduler_client.create_schedule(
            Name=schedule_name,
            GroupName='AmazonSageMakerAIPipelines',
            ScheduleExpression='cron(0 2 ? * MON *)',
            FlexibleTimeWindow={
                'Mode': 'OFF'
            },
            Target={
                'Arn': f'arn:aws:sagemaker:{region}:{account_id}:pipeline/{pipeline_name}',
                'RoleArn': role,
                'SageMakerPipelineParameters': {
                    'PipelineParameterList': [
                        {
                            'Name': 'EdadInit',
                            'Value': '25'
                        },
                        {
                            'Name': 'EdadEnd',
                            'Value': '65'
                        }
                    ]
                }
            },
            State='ENABLED',
            Description=f'Scheduled training for {pipeline_name}'
        )
        
        print(f"✅ EventBridge Schedule creado: {schedule_name}")
        print("📅 Se ejecutará todos los lunes a las 2 AM UTC")
        return response
        
    except Exception as e:
        print(f"❌ Error creando schedule: {e}")
        print("\n💡 Alternativas:")
        print("1. Usar el botón 'Scheduler' en SageMaker Studio")
        print("2. Crear desde AWS Console > EventBridge > Schedules")
        return None

create_eventbridge_schedule()